# 3. Pace components

## 3.1. Setup and load augmented data

Load events, transitions, and possession context exported by notebook 02 section 2.8, plus teams and games. Transitions retain recorded and modeled endpoint clocks.

In [52]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import is_integer_dtype

pd.set_option('display.max_columns', None)

ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

augmented_events_df = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet")
transitions_df = pd.read_parquet(PROCESSED_DATA_DIR / "event_transitions.parquet")
possession_context_df = pd.read_parquet(PROCESSED_DATA_DIR / "possession_context.parquet")
teams_df = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games_df = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")

# Calculate pace on a copy of the augmented transitions.
pace_df = transitions_df.copy()

## 3.2. Define eligible pace intervals

### Recorded elapsed time

Subtract the end clock from the start clock. Preserve recorded timestamps.

In [53]:
# Recorded clocks count down within each period.
pace_df["elapsed_seconds"] = (
    pace_df["start_clock_seconds"] - pace_df["end_clock_seconds"]
)

### Modeled elapsed time

Subtract exported modeled endpoint clocks (clocks with fractions of seconds)

In [54]:
# Consume modeled clocks from notebook 02; do not recompute event timing.
pace_df["modeled_elapsed_seconds"] = (
    pace_df["start_modeled_clock_seconds"] - pace_df["end_modeled_clock_seconds"]
)

## 3.3. Calculate distance and speed components

### Displacement and distance

Calculate signed x/y displacement and straight-line distance in rink feet. Increasing x is north.

In [55]:
# Straight-line distances between recorded positions, in rink feet.
# Distance is independent of recorded or modeled timing.
pace_df["dx_ft"] = (pace_df["end_x"] - pace_df["start_x"])
pace_df["dy_ft"] = (pace_df["end_y"] - pace_df["start_y"])
pace_df["distance_ft"] = np.hypot(pace_df["dx_ft"], pace_df["dy_ft"])

### Directional speeds

Calculate total, east-west, north-south, and north-only speed in feet per second using positive modeled durations. Backward movement has zero north-only speed.

In [56]:
# Calculate individual speeds from positive modeled durations.
# Identical-position ties retain zero duration and missing individual speed.
duration = pace_df["modeled_elapsed_seconds"].where(pace_df["modeled_elapsed_seconds"].gt(0))
pace_df["speed_total_ft_s"] = pace_df["distance_ft"] / duration
pace_df["speed_ew_ft_s"] = pace_df["dy_ft"].abs() / duration
pace_df["speed_ns_ft_s"] = pace_df["dx_ft"].abs() / duration
pace_df["speed_n_ft_s"] = pace_df["dx_ft"].clip(lower=0) / duration

speed_columns = [
    "speed_total_ft_s", "speed_ew_ft_s", "speed_ns_ft_s", "speed_n_ft_s",
]

display(pace_df[[
    "start_event_id", "end_event_id", "elapsed_seconds", "modeled_elapsed_seconds", "dx_ft", "dy_ft",
    "distance_ft", *speed_columns,
]].head(10))


,start_event_id,end_event_id,elapsed_seconds,modeled_elapsed_seconds,dx_ft,dy_ft,distance_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s
0,1,2,2.0,2.0,7.0,-3.0,7.615773,3.807887,1.5,3.5,3.5
1,2,3,1.0,1.0,18.0,-12.0,21.633308,21.633308,12.0,18.0,18.0
2,3,4,2.0,2.0,6.0,0.0,6.0,3.0,0.0,3.0,3.0
3,5,6,1.0,1.0,-10.0,5.0,11.18034,11.18034,5.0,10.0,0.0
4,7,8,2.0,2.0,24.0,-25.0,34.655447,17.327723,12.5,12.0,12.0
5,8,9,1.0,1.0,32.0,-6.0,32.557641,32.557641,6.0,32.0,32.0
6,9,10,0.0,0.5,41.0,38.0,55.901699,111.803399,76.0,82.0,82.0
7,10,11,4.0,3.5,-5.0,-18.0,18.681542,5.337583,5.142857,1.428571,0.0
8,11,12,0.0,0.5,3.0,13.0,13.341664,26.683328,26.0,6.0,6.0
9,13,14,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>


## 3.4. Calculate zonal components

### Zone transitions

Assign each transition to its destination zone, including its full distance and time when crossing a blue line. Segment identifiers are scoped to each possession. Shared boundary events connect adjacent intervals without duplicating distance or time.

In [57]:
# Keep aggregation inputs separate from the transition-level pace table.
aggregation_df = pace_df.copy()
aggregation_df["distance_ew_ft"] = aggregation_df["dy_ft"].abs()
aggregation_df["distance_ns_ft"] = aggregation_df["dx_ft"].abs()
aggregation_df["distance_n_ft"] = aggregation_df["dx_ft"].clip(lower=0)

# Zone labels use the possession team's increasing-x attacking direction.
def zone_from_x(x):
    zone = pd.Series(pd.NA, index=x.index, dtype="string")
    zone.loc[x.between(0, 75, inclusive="left")] = "DZ"
    zone.loc[x.between(75, 125)] = "NZ"
    zone.loc[x.gt(125) & x.le(200)] = "OZ"
    return zone

zonal_transitions_df = aggregation_df.sort_values("end_event_id", kind="stable").copy()
zonal_transitions_df["start_zone"] = zone_from_x(zonal_transitions_df["start_x"])
zonal_transitions_df["zone"] = zone_from_x(zonal_transitions_df["end_x"])
# Each interval belongs to its destination zone. A crossing interval
# enters the next segment; its end event anchors the following interval.
previous_zone = zonal_transitions_df.groupby("possession_id", sort=False)["zone"].shift()
new_segment = zonal_transitions_df["zone"].ne(previous_zone).fillna(True)
zonal_transitions_df["zone_segment_id"] = (
    new_segment.groupby(zonal_transitions_df["possession_id"], sort=False)
    .cumsum().astype("Int64")
)
if zonal_transitions_df[["start_zone", "zone"]].isna().any().any():
    raise ValueError("Missing transition zone; fix coordinates before calculating pace.")

### Zonal pace

Sum component distances and modeled time by zone segment.

In [58]:
distance_columns = ["distance_ft", "distance_ew_ft", "distance_ns_ft", "distance_n_ft"]
component_columns = dict(zip(distance_columns, [
    "speed_total_ft_s", "speed_ew_ft_s", "speed_ns_ft_s", "speed_n_ft_s",
]))

# Pool distances and modeled time; preserve recorded totals for comparison.
def summarize_pace(intervals, keys):
    work = intervals.copy()
    work["same_second_transitions"] = work["elapsed_seconds"].eq(0).astype("Int64")
    totals = work.groupby(keys, dropna=False, sort=False).agg(
        transitions=("end_event_id", "size"),
        sequences=("possession_id", "nunique"),
        same_second_transitions=("same_second_transitions", "sum"),
        elapsed_seconds=("elapsed_seconds", "sum"),
        modeled_elapsed_seconds=("modeled_elapsed_seconds", "sum"),
        **{column: (column, "sum") for column in distance_columns},
    ).reset_index()
    denominator = totals["modeled_elapsed_seconds"].where(
        totals["modeled_elapsed_seconds"].gt(0)
    )
    for distance, speed in component_columns.items():
        totals[speed] = totals[distance] / denominator
    return totals

zonal_pace_df = summarize_pace(zonal_transitions_df, [
    "source_dataset", "game_id", "period", "possession_id",
    "possession_team_id", "zone_segment_id", "zone",
])
display(zonal_pace_df.head(10))

,source_dataset,game_id,period,possession_id,possession_team_id,zone_segment_id,zone,transitions,sequences,same_second_transitions,elapsed_seconds,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s
0,NWHL,3,1,1,1,1,NZ,2,1,0,3.0,3.0,29.249081,15.0,25.0,25.0,9.749694,5.0,8.333333,8.333333
1,NWHL,3,1,1,1,2,OZ,1,1,0,2.0,2.0,6.0,0.0,6.0,6.0,3.0,0.0,3.0,3.0
2,NWHL,3,1,2,1,1,OZ,1,1,0,1.0,1.0,11.18034,5.0,10.0,0.0,11.18034,5.0,10.0,0.0
3,NWHL,3,1,3,5,1,NZ,1,1,0,2.0,2.0,34.655447,25.0,24.0,24.0,17.327723,12.5,12.0,12.0
4,NWHL,3,1,3,5,2,OZ,4,1,2,5.0,5.5,120.482546,75.0,81.0,76.0,21.905918,13.636364,14.727273,13.818182
5,NWHL,3,1,4,1,1,DZ,3,1,2,3.0,3.0,57.654288,33.0,47.0,47.0,19.218096,11.0,15.666667,15.666667
6,NWHL,3,1,5,5,1,OZ,1,1,0,2.0,2.0,3.162278,1.0,3.0,3.0,1.581139,0.5,1.5,1.5
7,NWHL,3,1,6,1,1,DZ,3,1,1,4.0,4.5,95.155822,73.0,60.0,60.0,21.145738,16.222222,13.333333,13.333333
8,NWHL,3,1,7,5,1,NZ,1,1,0,2.0,2.0,12.165525,2.0,12.0,12.0,6.082763,1.0,6.0,6.0
9,NWHL,3,1,7,5,2,OZ,1,1,0,2.0,2.0,38.0,0.0,38.0,38.0,19.0,0.0,19.0,19.0


## 3.5. Aggregate pace

### Possession pace

Pool distance and modeled duration, keeping final movements and recorded totals for comparison. Zero-duration or transition-free possessions have missing pace.

In [59]:
# Include possessions with no transitions so coverage remains visible.
possession_metadata_df = possession_context_df[
    ["possession_id", "source_dataset", "game_id", "period", "possession_team_id", "event_count"]
].rename(columns={"event_count": "events"}).copy()
assert possession_metadata_df["possession_id"].is_unique
assert set(possession_metadata_df["possession_id"]) == set(augmented_events_df["possession_id"])
possession_metadata_df = possession_metadata_df[[
    "possession_id", "source_dataset", "game_id", "period", "possession_team_id", "events",
]]
possession_pace_df = possession_metadata_df.merge(
    summarize_pace(aggregation_df, ["possession_id"]).drop(columns="sequences"),
    on="possession_id", how="left", validate="one_to_one",
)
for column in ["transitions", "same_second_transitions"]:
    possession_pace_df[column] = possession_pace_df[column].fillna(0).astype("Int64")
possession_pace_df["pace_status"] = pd.Series("eligible", index=possession_pace_df.index, dtype="string")
possession_pace_df.loc[possession_pace_df["modeled_elapsed_seconds"].eq(0), "pace_status"] = "zero_duration"
possession_pace_df.loc[possession_pace_df["transitions"].eq(0), "pace_status"] = "no_transitions"

# Opponent identity is based on the possession owner, not a contextual event team.
game_lookup = games_df.set_index("game_id")
home_ids = possession_pace_df["game_id"].map(game_lookup["home_team_id"])
away_ids = possession_pace_df["game_id"].map(game_lookup["away_team_id"])
possession_pace_df["opponent_team_id"] = away_ids.where(
    possession_pace_df["possession_team_id"].eq(home_ids), home_ids
).astype("Int64")

display(possession_pace_df.groupby(["source_dataset", "pace_status"]).size().unstack(fill_value=0))

pace_status,eligible,no_transitions,zero_duration
source_dataset,,,
NWHL,5697,742,776
Olympics 2022,2198,369,215
Womens,4861,609,579


### Team pace by game and period

Calculate the pace for both teams across all situations. Pool component distances and modeled movement time for every period, including overtime. Derive full-game totals from those same period totals using period 0.

Pace is summed distance divided by summed modeled time.

In [60]:
def build_game_team_pace(possessions: pd.DataFrame, games: pd.DataFrame) -> pd.DataFrame:
    """Pool distance and modeled time for both teams, by period and full game."""
    allocations = ["modeled_elapsed_seconds", *distance_columns]
    keys = ["game_id", "team_id", "period"]

    # Prepare possession totals; no-transition possessions contribute zero.
    work = possessions.rename(columns={"possession_team_id": "team_id"}).copy()
    work.loc[work["transitions"].eq(0), allocations] = 0
    pooled = work.groupby(keys, as_index=False)[allocations].sum()

    # Include both teams in every recorded period, then sum full-game totals.
    game_teams = games.melt(
        id_vars="game_id", value_vars=["away_team_id", "home_team_id"],
        value_name="team_id",
    )[["game_id", "team_id"]]
    game_periods = work[["game_id", "period"]].drop_duplicates()
    periods = game_teams.merge(game_periods, on="game_id").merge(
        pooled, on=keys, how="left", validate="one_to_one",
    )
    periods[allocations] = periods[allocations].fillna(0)
    periods = periods.sort_values(keys).reset_index(drop=True)
    full = game_teams.merge(
        periods.groupby(["game_id", "team_id"], as_index=False)[allocations].sum(),
        on=["game_id", "team_id"], how="left", validate="one_to_one",
    )
    full[allocations] = full[allocations].fillna(0)
    full["period"] = 0
    full = full.sort_values(["game_id", "team_id"]).reset_index(drop=True)
    result = pd.concat([full, periods], ignore_index=True)

    # Calculate speeds from pooled totals; zero exposure keeps missing speeds.
    modeled_time = result["modeled_elapsed_seconds"].where(result["modeled_elapsed_seconds"].gt(0))
    for distance, speed in component_columns.items():
        result[speed] = result[distance] / modeled_time
    return result[[*keys, *allocations, *component_columns.values()]]

game_team_pace_df = build_game_team_pace(possession_pace_df, games_df)
team_names = teams_df.set_index("team_id")["team_name"]
display(game_team_pace_df.assign(team_name=game_team_pace_df["team_id"].map(team_names)))

,game_id,team_id,period,modeled_elapsed_seconds,distance_ft,distance_ew_ft,distance_ns_ft,distance_n_ft,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s,team_name
0,1,2,0,1092.0,18417.664364,9641.0,13137.0,10264.0,16.865993,8.828755,12.03022,9.399267,Buffalo Beauts
1,1,3,0,1425.5,25118.561375,13288.0,17724.0,13158.0,17.620878,9.321642,12.433532,9.230445,Connecticut Whale
2,2,4,0,949.833333,17272.60877,9482.0,12008.0,8925.0,18.184884,9.982804,12.642218,9.396385,Metropolitan Riveters
3,2,6,0,1361.666667,23072.606484,12501.0,16194.0,11424.0,16.944387,9.180661,11.892778,8.389718,Toronto Six
4,3,1,0,1400.25,24820.797904,13036.0,17861.0,12595.0,17.725976,9.309766,12.755579,8.994822,Boston Pride
...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,34,16,4,398.0,7645.436331,4232.0,5336.0,3586.0,19.209639,10.633166,13.407035,9.01005,Finland (O19)
278,34,20,1,554.0,10113.461628,5439.0,7024.0,4707.0,18.255346,9.81769,12.6787,8.49639,United States (O19)
279,34,20,2,444.0,7162.969059,3843.0,5004.0,3390.0,16.132813,8.655405,11.27027,7.635135,United States (O19)
280,34,20,3,583.5,9808.533025,5307.0,6922.0,5067.0,16.809825,9.095116,11.862896,8.683805,United States (O19)


## 3.6. Calculate final whole-game polygrid pace

Allocate each valid movement's distance and modeled time equally among intersected 5 ? 5 ft cells. Pool all situations and periods by game, possession team, and cell. Keep original attacking coordinates for both teams. Only final game maps and cell geometry are exported.

### Rink grid

Use the previous implementation's 668-cell layout: a 40 ? 17 grid with three cells removed at each corner. This approximates rounded corners; the paper does not specify the exact cuts. Edge and corner touches count as intersections, including stationary positions on cell boundaries. Coordinates remain in the attacking team's perspective for both roles.

In [61]:
from scipy.ndimage import gaussian_filter

CELL_SIZE_FT = 5
GRID_ROWS, GRID_COLUMNS = 17, 40
GAUSSIAN_SIGMA = 2

corner_cuts = {(0, 0), (1, 0), (0, 1)}
removed_cells = {
    (GRID_COLUMNS - 1 - col if flip_x else col,
     GRID_ROWS - 1 - row if flip_y else row)
    for col, row in corner_cuts
    for flip_x in [False, True] for flip_y in [False, True]
}
polygrid_cells_df = pd.DataFrame([
    {"grid_column": col, "grid_row": row,
     "x_min_ft": col * CELL_SIZE_FT, "x_max_ft": (col + 1) * CELL_SIZE_FT,
     "y_min_ft": row * CELL_SIZE_FT, "y_max_ft": (row + 1) * CELL_SIZE_FT}
    for row in range(GRID_ROWS) for col in range(GRID_COLUMNS)
    if (col, row) not in removed_cells
])
polygrid_cells_df.insert(0, "cell_id", np.arange(len(polygrid_cells_df)))

# Slab intersection includes cells touched only at an edge or corner.
cell_lower = polygrid_cells_df[["x_min_ft", "y_min_ft"]].to_numpy(float)
cell_upper = polygrid_cells_df[["x_max_ft", "y_max_ft"]].to_numpy(float)

def intersected_cells(x0, y0, x1, y1):
    start = np.array([x0, y0], dtype=float)
    delta = np.array([x1 - x0, y1 - y0], dtype=float)
    enter = np.zeros(len(polygrid_cells_df))
    leave = np.ones(len(polygrid_cells_df))
    inside = np.ones(len(polygrid_cells_df), dtype=bool)
    for axis in range(2):
        if abs(delta[axis]) < 1e-12:
            inside &= (start[axis] >= cell_lower[:, axis] - 1e-10)
            inside &= (start[axis] <= cell_upper[:, axis] + 1e-10)
        else:
            first = (cell_lower[:, axis] - start[axis]) / delta[axis]
            last = (cell_upper[:, axis] - start[axis]) / delta[axis]
            enter = np.maximum(enter, np.minimum(first, last))
            leave = np.minimum(leave, np.maximum(first, last))
    return np.flatnonzero(inside & (enter <= leave + 1e-10)).tolist()

# Check the generated grid geometry before allocating movements.
assert len(polygrid_cells_df) == 668
assert polygrid_cells_df["cell_id"].is_unique
assert not polygrid_cells_df.duplicated(["grid_row", "grid_column"]).any()
assert polygrid_cells_df["grid_row"].between(0, GRID_ROWS - 1).all()
assert polygrid_cells_df["grid_column"].between(0, GRID_COLUMNS - 1).all()
for axis, position in [("x", "grid_column"), ("y", "grid_row")]:
    assert polygrid_cells_df[f"{axis}_min_ft"].eq(polygrid_cells_df[position] * CELL_SIZE_FT).all()
    assert polygrid_cells_df[f"{axis}_max_ft"].eq((polygrid_cells_df[position] + 1) * CELL_SIZE_FT).all()

### Temporary transition-to-cell allocations

Keep valid zero-duration movements and stationary intervals. Allocations exist only in memory to build final game maps and verify conservation. Paths entirely inside removed corners are reported as unassigned. No chronological replay data is exported.

In [62]:
# Find the grid cells touched by each movement.
polygrid_intervals = aggregation_df.copy()
polygrid_intervals["cell_ids"] = [
    intersected_cells(movement.start_x, movement.start_y, movement.end_x, movement.end_y)
    for movement in polygrid_intervals.itertuples(index=False)
]

# Separate movements with no grid coverage from those we can allocate.
cell_counts = polygrid_intervals["cell_ids"].str.len()
has_grid_coverage = cell_counts.gt(0)
polygrid_unassigned_df = polygrid_intervals.loc[~has_grid_coverage].drop(columns="cell_ids")
covered_transitions = polygrid_intervals.loc[has_grid_coverage].copy()


In [63]:
# Divide each movement equally among the cells it touches.
covered_transitions["share"] = 1 / cell_counts.loc[has_grid_coverage]
reference_columns = [
    "source_dataset", "game_id", "period", "possession_id", "possession_team_id",
    "start_event_id", "end_event_id",
]
measurement_columns = ["elapsed_seconds", "modeled_elapsed_seconds", *distance_columns]
contribution_columns = [*reference_columns, "cell_ids", "share", *measurement_columns]

# Expand the list of cells into one row per movement and cell.
polygrid_contributions_df = covered_transitions[contribution_columns].explode("cell_ids")
polygrid_contributions_df = polygrid_contributions_df.rename(columns={"cell_ids": "cell_id"})
polygrid_contributions_df = polygrid_contributions_df.reset_index(drop=True)
polygrid_contributions_df["cell_id"] = polygrid_contributions_df["cell_id"].astype(int)


In [64]:
# Apply the same fraction to distance and time, preserving zero-time distances.
for column in measurement_columns:
    polygrid_contributions_df[column] *= polygrid_contributions_df["share"]


polygrid_contributions_df = polygrid_contributions_df.sort_values(
    ["end_event_id", "cell_id"], kind="stable"
).reset_index(drop=True)


### Final game maps

For each game and both participating teams, smooth the four pooled distance arrays and pooled modeled time separately on a 17 x 40 grid. Use Gaussian sigma 2 cells (10 feet). Divide smoothed distance by smoothed time only where original exposure is positive. Missing speeds remain missing and removed corner cells are excluded.

In [65]:
def smooth_team_polygrid(totals):
    """Smooth one team's game distances and time, keeping original exposure."""
    rows = totals["grid_row"].to_numpy(int)
    columns = totals["grid_column"].to_numpy(int)

    # Place cell exposure on the rink grid and smooth it once.
    time_grid = np.zeros((GRID_ROWS, GRID_COLUMNS))
    time_grid[rows, columns] = totals["modeled_elapsed_seconds"].to_numpy(float)
    smoothed_time = gaussian_filter(time_grid, sigma=GAUSSIAN_SIGMA, mode="constant", truncate=4)
    result = totals[["cell_id", "modeled_elapsed_seconds"]].copy()

    # Smooth each distance component, then divide by smoothed time.
    for distance, speed in component_columns.items():
        distance_grid = np.zeros_like(time_grid)
        distance_grid[rows, columns] = totals[distance].to_numpy(float)
        smoothed_distance = gaussian_filter(distance_grid, sigma=GAUSSIAN_SIGMA, mode="constant", truncate=4)
        speed_grid = np.divide(
            smoothed_distance, smoothed_time,
            out=np.full_like(time_grid, np.nan),
            where=(time_grid > 0) & (smoothed_time > 0),
        )
        result[speed] = speed_grid[rows, columns]
    return result


def build_game_polygrids(cells, contributions, games):
    """Pool whole-game cell totals and combine both teams' smoothed maps."""
    allocations = ["modeled_elapsed_seconds", *distance_columns]
    keys = ["game_id", "team_id", "cell_id"]

    # Pool movements by game, possession team, and cell.
    pooled = contributions.rename(columns={"possession_team_id": "team_id"})
    pooled = pooled.groupby(keys, as_index=False)[allocations].sum()

    # Complete both teams' grids, including cells with no observed movement.
    game_teams = games.melt(
        id_vars="game_id", value_vars=["away_team_id", "home_team_id"],
        value_name="team_id",
    )[["game_id", "team_id"]].sort_values("game_id", kind="stable")
    complete_grid = game_teams.merge(cells, how="cross")
    complete_grid = complete_grid.merge(pooled, on=keys, how="left", validate="one_to_one")
    complete_grid[allocations] = complete_grid[allocations].fillna(0)

    # Smooth each game/team map and combine the final cell records.
    maps = []
    for (game_id, team_id), totals in complete_grid.groupby(["game_id", "team_id"], sort=False):
        team_map = smooth_team_polygrid(totals)
        team_map["game_id"] = game_id
        team_map["team_id"] = team_id
        maps.append(team_map)
    output_columns = [*keys, "modeled_elapsed_seconds", *component_columns.values()]
    return pd.concat(maps, ignore_index=True)[output_columns]


game_polygrid_df = build_game_polygrids(polygrid_cells_df, polygrid_contributions_df, games_df)
display(game_polygrid_df.head())

,game_id,team_id,cell_id,modeled_elapsed_seconds,speed_total_ft_s,speed_ew_ft_s,speed_ns_ft_s,speed_n_ft_s
0,1,3,0,0.2,13.464336,7.901361,8.857936,7.081575
1,1,3,1,1.105303,14.422946,8.136468,9.872034,8.041200
2,1,3,2,2.325794,15.363932,8.344988,10.862076,9.042215
3,1,3,3,2.363125,16.316013,8.515471,11.857936,10.115268
4,1,3,4,5.604383,17.345557,8.620920,12.956422,11.322526


## 3.7. Build database tables

In [66]:
possession_columns = [
    "possession_id", "game_id", "period", "possession_team_id", "source_dataset",
    "start_event_id", "end_event_id", "start_clock_seconds", "end_clock_seconds", "event_count",
    "elapsed_seconds", "modeled_elapsed_seconds", "speed_total_ft_s", "pace_status", "outcome",
    "home_score", "away_score", "home_skaters", "away_skaters", "contains_goal",
]
possessions_df = possession_context_df.merge(
    possession_pace_df[["possession_id", "elapsed_seconds", "modeled_elapsed_seconds", "speed_total_ft_s", "pace_status"]],
    on="possession_id", how="left", validate="one_to_one",
)[possession_columns].sort_values("possession_id").reset_index(drop=True)
geometry_columns = ["x_min_ft", "x_max_ft", "y_min_ft", "y_max_ft"]
polygrid_cells_df[geometry_columns] = polygrid_cells_df[geometry_columns].astype("Float64")
for frame in [possessions_df, game_team_pace_df, game_polygrid_df, polygrid_cells_df]:
    for column in frame.columns:
        if column.endswith("_id") or column in {"period", "event_count", "home_score", "away_score", "home_skaters", "away_skaters", "grid_row", "grid_column"}:
            frame[column] = frame[column].astype("Int64")
        elif column == "contains_goal":
            frame[column] = frame[column].astype("boolean")
        elif column in {"source_dataset", "pace_status", "outcome"}:
            frame[column] = frame[column].astype("string")
        else:
            frame[column] = frame[column].astype("Float64")

database_tables = {
    "possessions": possessions_df,
    "game_team_pace": game_team_pace_df,
    "polygrid_cells": polygrid_cells_df,
    "game_polygrid": game_polygrid_df,
}

## 3.8. Export pace files


In [67]:
pace_exports = {
    "pace_transitions": pace_df,
    "pace_zonal_transitions": zonal_transitions_df,
    "pace_zonal_segments": zonal_pace_df,
    "pace_sequences": possession_pace_df,
    **database_tables,
}

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, table in pace_exports.items():
    table.to_parquet(
        PROCESSED_DATA_DIR / f"{name}.parquet",
        engine="pyarrow", compression="snappy", index=False,
    )

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in pace_exports.items()
]))


,table,rows,columns
0,pace_transitions,63298,24
1,pace_zonal_transitions,63298,30
2,pace_zonal_segments,21789,20
3,pace_sequences,16046,20
4,possessions,16046,20
5,game_team_pace,282,12
6,polygrid_cells,668,7
7,game_polygrid,45424,8
